# Day I9 — IMU의 원리: 가속도계는 중력도 재고, 자이로는 흘러간다

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I9.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I9.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 정지 자세의 기울기, 그리고 움직일 때의 오류

In [ ]:
import numpy as np

rng = np.random.default_rng(9)
fs = 100
true_tilt = 20.0                     # 정강이가 앞으로 20° 기울어 서 있다(가상)
th = np.deg2rad(true_tilt)
n = 2 * fs                           # 2초 기록
# 센서 x축 = 정강이 앞쪽, z축 = 정강이 장축(위쪽). 정지 상태에서는 중력만 보인다(단위 g)
ax = np.sin(th) + rng.normal(0, 0.01, n)
az = np.cos(th) + rng.normal(0, 0.01, n)
tilt = np.rad2deg(np.arctan2(ax, az))
print(f"정지: 가속도 크기 평균 {np.mean(np.hypot(ax, az)):.3f} g, 추정 기울기 {tilt.mean():.2f}° (참값 {true_tilt}°)")

# 같은 자세에서 센서 x축 방향으로 0.3 g의 선가속도가 더해지는 순간(예: 발을 내딛는 중)
tilt_move = np.rad2deg(np.arctan2(ax + 0.3, az))
print(f"움직임: 추정 기울기 {tilt_move.mean():.1f}° → 선가속도가 기울기처럼 읽힌다")

### 예시 2 — bias가 있는 자이로를 60초 적분하기

In [ ]:
import numpy as np

rng = np.random.default_rng(90)
fs = 100
t = np.arange(0, 60, 1 / fs)          # 60초
true_rate = np.zeros_like(t)          # 실제로는 전혀 돌지 않는다(가만히 놓인 센서)
for bias in [0.1, 0.5, 1.0]:          # 자이로의 영점 어긋남(°/s)
    gyro = true_rate + bias + rng.normal(0, 0.5, len(t))   # bias + 백색 잡음 0.5 °/s
    angle = np.cumsum(gyro) / fs                           # 적분 → 각도(°)
    e = [angle[int(s * fs) - 1] for s in (10, 30, 60)]
    print(f"bias {bias:.1f} °/s → 각도 오차 10 s {e[0]:5.1f}°, 30 s {e[1]:5.1f}°, 60 s {e[2]:5.1f}°")

### 예시 3 — 센서를 비뚤게 붙이면 같은 회전이 다른 숫자로 읽힌다

In [ ]:
import numpy as np

# 정강이 분절 좌표: x = 앞, y = 내외측 축, z = 장축(위)
w_anat = np.array([0.0, 300.0, 0.0])        # 시상면 회전 300 °/s → 내외측 축(y) 성분만 있다

def rot_z(deg):                              # 장축(z) 둘레로 센서를 돌려 붙였을 때의 변환
    c, s = np.cos(np.deg2rad(deg)), np.sin(np.deg2rad(deg))
    return np.array([[c, s, 0], [-s, c, 0], [0, 0, 1]])

for mis in [0, 10, 20, 30]:
    w_sensor = rot_z(mis) @ w_anat           # 센서가 자기 축으로 읽는 값
    print(f"장축 둘레 {mis:2d}° 어긋남 → 센서 x {w_sensor[0]:6.1f}, y {w_sensor[1]:6.1f}, "
          f"z {w_sensor[2]:4.1f} °/s | 크기 {np.linalg.norm(w_sensor):.1f}")

### 예시 4 — 실제 자료: 세 축 크기의 중앙값은 1 g 근처

**데이터:** Roggen D, Plotnik M, Hausdorff J. *Daphnet Freezing of Gait* [Dataset]. UCI Machine Learning Repository; 2010. [doi:10.24432/C56K78](https://doi.org/10.24432/C56K78). 라이선스 CC BY 4.0. 사용 시 Bächlin 2010 인용이 요구됩니다. [UCI 페이지](https://archive.ics.uci.edu/dataset/245/daphnet+freezing+of+gait)에서 zip을 받아 풀고, `dataset/S01R01.txt`를 `data/daphnet/`에 둡니다. (zip 안에 든 논문 PDF는 재배포하지 않습니다.) 파일 구조: 머리글 없음, 공백 구분 11열, 1열 시간(ms), 2–4열 발목(shank)의 앞뒤·수직·좌우 가속도(mg), 5–7열 허벅지, 8–10열 엉덩이 높이 센서, 11열 라벨(0 = 실험 외, 1 = 실험 중 FOG 아님, 2 = FOG), 64 Hz.

In [ ]:
import numpy as np

fs = 64                                           # Daphnet 표본화 주파수(데이터 문서)
d = np.loadtxt("data/daphnet/S01R01.txt")         # 머리글 없음, 공백 구분 11열
label = d[:, 10]                                  # 0 = 실험 외, 1 = FOG 아님, 2 = FOG
shank = d[:, 1:4]                                 # 2–4열: 발목(shank) 전후·수직·좌우, 단위 mg
on = label == 1                                   # 실험 중이고 FOG가 아닌 구간만
mag = np.linalg.norm(shank[on], axis=1)
print(f"기록 길이 {len(d) / fs / 60:.1f}분, 라벨 1 구간 {on.sum() / fs / 60:.1f}분")
print(f"축별 평균(mg): 전후 {shank[on, 0].mean():.0f}, 수직 {shank[on, 1].mean():.0f}, 좌우 {shank[on, 2].mean():.0f}")
print(f"세 축 크기의 중앙값 {np.median(mag):.0f} mg (1 g = 1000 mg)")